# Migration Validation Metrics (DPC1.0 → DPC2.0)

Collects metadata metrics for the Hive tables on both platforms and compares them, using the
same checks the NX1 MapR/HDFS → S3 migration DAG runs after each copy. It works no matter how the
data was copied. Nothing is changed on either platform: the collector only runs `SHOW`,
`DESCRIBE`, `SELECT COUNT(*)` and storage listings.

### Where each side is collected

- **DPC1.0 (source)** is collected **outside this notebook**: there is no SSH from JupyterHub to the
  edge node. Run `migration_metrics_collector.py` (or `migration_metrics_collector_v2.py`, which also
  copies the result to S3) on the DPC1.0 edge node with `spark-submit`. Put the resulting JSON in an
  S3 location that both platforms can reach and set `SOURCE_SNAPSHOT_PATH` below to it.
- **DPC2.0 (destination)** is collected in this JupyterHub session's Spark, for the same databases and
  tables that the source snapshot recorded.

Both sides are measured by the same `migration_metrics_collector.py`, so any difference in the
report comes from the data, not from the tooling. **Upload it to the same JupyterHub folder as
this notebook.**

### What is checked per table

| Metric | How it is collected | Mismatch → |
|---|---|---|
| Row count (total and per partition) | `COUNT(*)` grouped by the partition columns | **FAIL** |
| Schema: non-partition columns, name + type | `spark.table(...).schema` | **FAIL** |
| Partition columns, name + type | `DESCRIBE TABLE EXTENDED` + schema | **FAIL** |
| Registered partitions | `SHOW PARTITIONS` | WARN |
| Data files and bytes (total and per partition) | listing under the table `LOCATION`, skipping hidden files (`_SUCCESS`, `.crc`, `_temporary`, …) | WARN |
| Partition directories that hold data but are not registered | storage listing vs `SHOW PARTITIONS` | WARN (fix with `MSCK REPAIR TABLE`) |

Other statuses: `MISSING_ON_DEST` (table not found on DPC2.0), `EXTRA_ON_DEST` (table only on
DPC2.0), `ERROR` (metrics could not be collected), `SKIPPED_VIEW` (views hold no data).

### How to run

1. On the DPC1.0 edge node, run the source collector (V1 or V2) and make sure its JSON ends up in S3.
2. Fill in the configuration cell: `SOURCE_SNAPSHOT_PATH`, `OUTPUT_DIR` and `RUN_LABEL`. No
   credentials are stored in the notebook, and nothing connects to the edge node.
3. Run all cells.

The source snapshot is read from `SOURCE_SNAPSHOT_PATH`, checked, and copied to `source_metrics.json`
under `OUTPUT_DIR/RUN_LABEL`. The DPC2.0 metrics are saved as `dest_metrics.json` there. The comparison
only reads those two files. To compare again without re-collecting DPC2.0 (e.g. to regenerate the
report), set `COLLECT_DEST = False`.

### Outputs

```
OUTPUT_DIR/RUN_LABEL/
├── source_metrics.json             # raw metrics, one record per table
├── dest_metrics.json
└── report_<UTC timestamp>/
    ├── summary.csv                 # one row per table, with status and every metric
    ├── partition_diffs.csv         # every partition that differs
    └── report.html
```

### Limits

- Storage metrics only cover files under the table `LOCATION`. Row counts and `SHOW PARTITIONS` still
  include partitions that live elsewhere.
- On Iceberg tables, the storage listing also counts files left behind by expired snapshots that
  have not been cleaned up yet, and `SHOW PARTITIONS` is skipped.
- Metrics are a point-in-time snapshot. If tables are still being written to, collect both sides
  again close together.
- `COUNT(*)` on Parquet/ORC is answered from file footers. Text tables are scanned in full.

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────────────
import os

# Groups the snapshots and reports of one validation exercise under OUTPUT_DIR/RUN_LABEL.
RUN_LABEL = "dpc1-to-dpc2"

# Where snapshots and reports are written: an s3a:// path or a local directory.
OUTPUT_DIR = os.environ.get("MIGRATION_VALIDATION_DIR", "s3a://CHANGE_ME/migration-validation")

# S3 location of the source snapshot JSON: the file written by migration_metrics_collector.py (uploaded
# by hand) or by migration_metrics_collector_v2.py (copied to S3 for you) on the DPC1.0 edge node.
# The databases and tables to validate come from this snapshot, so both sides always match.
SOURCE_SNAPSHOT_PATH = os.environ.get(
    "SOURCE_SNAPSHOT_PATH", "s3a://CHANGE_ME/migration-validation/dpc1-to-dpc2/source_metrics.json"
)

# True: collect DPC2.0 now. False: reuse the dest_metrics.json already under OUTPUT_DIR/RUN_LABEL.
COLLECT_DEST = True

# "catalog" is prefixed to every table name on DPC2.0; empty = default catalog.
DEST = {"label": "DPC2.0", "catalog": ""}

COLLECTOR_FILE = "migration_metrics_collector.py"  # relative to this notebook

# Lists every data file under each table location. If that is too slow for very large tables,
# turn it off: row counts, partitions and schema are still compared.
COLLECT_STORAGE_METRICS = True
PARALLELISM = 8  # tables collected concurrently, on each side
REPORT_MAX_PARTITION_ROWS = 500  # cap for the HTML report only; partition_diffs.csv has every row

assert "CHANGE_ME" not in OUTPUT_DIR, "Set OUTPUT_DIR (or the MIGRATION_VALIDATION_DIR env var)"
assert "CHANGE_ME" not in SOURCE_SNAPSHOT_PATH, "Set SOURCE_SNAPSHOT_PATH (or the SOURCE_SNAPSHOT_PATH env var)"

---
## Helpers

Nothing in this section touches the tables or the edge node. Run it, then move on.

In [ ]:
# ── Spark session, collector module, storage I/O ────────────────────────────────────
import html
import importlib
import json
import re
import sys
from datetime import datetime, timezone

import pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession

try:
    _ = spark
    print(f"Using existing Spark session (version {spark.version})")
except NameError:
    spark = SparkSession.builder.appName("migration-validation-metrics").enableHiveSupport().getOrCreate()
    print(f"Created Spark session (version {spark.version})")

COLLECTOR_PATH = os.path.abspath(COLLECTOR_FILE)
assert os.path.isfile(COLLECTOR_PATH), (
    f"{COLLECTOR_PATH} not found: upload migration_metrics_collector.py next to this notebook"
)
sys.path.insert(0, os.path.dirname(COLLECTOR_PATH))
import migration_metrics_collector as mmc  # noqa: E402

mmc = importlib.reload(mmc)

OUTPUT_ROOT = re.sub(r"^s3n?://", "s3a://", OUTPUT_DIR.rstrip("/"))
if "://" not in OUTPUT_ROOT:
    OUTPUT_ROOT = "file://" + os.path.abspath(OUTPUT_ROOT)
RUN_DIR = f"{OUTPUT_ROOT}/{RUN_LABEL}"


def snapshot_path(side):
    return f"{RUN_DIR}/{side}_metrics.json"


def _hpath(path):
    return spark._jvm.org.apache.hadoop.fs.Path(path)


def _fs(hpath):
    return hpath.getFileSystem(spark._jsc.hadoopConfiguration())


# Snapshots and reports go through the Hadoop FileSystem on the driver, so s3a:// and local
# paths work the same way and no executor ever needs to see a local path.
def write_text(path, text):
    hp = _hpath(path)
    out = _fs(hp).create(hp, True)
    try:
        out.write(bytearray(text.encode("utf-8")))
    finally:
        out.close()


def read_text(path):
    hp = _hpath(path)
    fs = _fs(hp)
    if not fs.exists(hp):
        return None
    buf = spark._jvm.java.io.ByteArrayOutputStream()
    stream = fs.open(hp)
    try:
        spark._jvm.org.apache.hadoop.io.IOUtils.copyBytes(stream, buf, 1 << 16, False)
    finally:
        stream.close()
    return buf.toString("UTF-8")


def dest_collector_config(src_snap):
    # Databases, tables and partition filters come from the source snapshot, so both sides cover the same set.
    return {
        "side": "dest",
        "label": DEST["label"],
        "catalog": DEST["catalog"],
        "run_label": RUN_LABEL,
        "database_pairs": src_snap["database_pairs"],
        "collect_storage_metrics": COLLECT_STORAGE_METRICS,
        "parallelism": PARALLELISM,
    }


def load_source_snapshot(path):
    """Read and check the source snapshot JSON. Returns (snapshot, raw text, normalized path)."""
    path = re.sub(r"^s3n?://", "s3a://", path.strip())
    if "://" not in path:
        path = "file://" + os.path.abspath(path)
    try:
        text = read_text(path)
    except Exception as e:
        raise RuntimeError(f"Could not read the source snapshot {path}: {str(e)[:500]}") from e
    if text is None:
        raise FileNotFoundError(f"No source snapshot at {path}: check the path and that the file was uploaded")
    try:
        snap = json.loads(text)
    except ValueError as e:
        raise ValueError(f"The source snapshot {path} is not valid JSON (partial upload?): {e}") from e
    problems = []
    if not isinstance(snap, dict):
        problems.append("the top level is not a JSON object")
    else:
        if snap.get("side") != "source":
            problems.append(f"'side' is {snap.get('side')!r}, expected 'source'")
        if not isinstance(snap.get("tables"), list):
            problems.append("'tables' is missing or not a list")
        if not snap.get("database_pairs"):
            problems.append("'database_pairs' is missing or empty")
    if problems:
        raise ValueError(f"{path} is not a collector snapshot: " + "; ".join(problems))
    return snap, text, path




print(f"Collector: {COLLECTOR_PATH}")
print(f"Snapshots and reports: {RUN_DIR}")

---
## Step 1: Load the source snapshot and collect DPC2.0

Reads the source snapshot from `SOURCE_SNAPSHOT_PATH`, checks that it is a valid collector snapshot,
and copies it to `source_metrics.json` under `OUTPUT_DIR/RUN_LABEL`. Then collects DPC2.0 for the
same database pairs and saves it as `dest_metrics.json`, overwriting the previous one. Nothing here
connects to the DPC1.0 edge node.

In [ ]:
# ── Source: read the snapshot from S3 ──
src_snapshot, _src_text, _src_path = load_source_snapshot(SOURCE_SNAPSHOT_PATH)
print(f"Source snapshot: {_src_path}")
print(f"  {src_snapshot['label']} collected {src_snapshot['collected_at']} "
      f"(Spark {src_snapshot.get('spark_version', '?')}), {len(src_snapshot['tables'])} record(s), "
      f"{len(src_snapshot['database_pairs'])} database pair(s)")
if _src_path != snapshot_path("source"):
    write_text(snapshot_path("source"), _src_text)
    print(f"  copied to {snapshot_path('source')}")

# ── Destination: collect in this session's Spark, for the same database pairs ──
if COLLECT_DEST:
    _dest_text = json.dumps(mmc.collect_side(spark, dest_collector_config(src_snapshot), emit=print))
    write_text(snapshot_path("dest"), _dest_text)
    print(f"[dest] snapshot saved: {snapshot_path('dest')}\n")
else:
    print(f"COLLECT_DEST is False: reusing {snapshot_path('dest')}")

---
## Step 2: Compare

Reads both snapshots back from `OUTPUT_DIR` and compares them table by table. Tables are
matched by name within each database pair.

In [ ]:
_texts = {side: read_text(snapshot_path(side)) for side in ("source", "dest")}
missing = [side for side, text in _texts.items() if text is None]
summary_df = partitions_df = None
if missing:
    print(f"Comparison skipped: no snapshot yet for {missing} under {RUN_DIR}. Collect it first.")
else:
    src_snapshot, dest_snapshot = json.loads(_texts["source"]), json.loads(_texts["dest"])
    for snap in (src_snapshot, dest_snapshot):
        print(f"{snap['side']:>6}: {snap['label']} collected {snap['collected_at']} "
              f"(Spark {snap['spark_version']}, Python {snap.get('python_version', '?')})")
    _rows, _diffs, _warnings = mmc.compare_snapshots(src_snapshot, dest_snapshot)
    for w in _warnings:
        print(f"WARNING: {w}")

    summary_cols = [
        "pair", "table", "status", "issues", "source_rows", "dest_rows", "source_partitions", "dest_partitions",
        "source_files", "dest_files", "source_bytes", "dest_bytes", "partition_issues", "schema_differences",
        "partition_schema_differences", "notes", "source_location", "dest_location",
    ]
    diff_cols = ["pair", "table", "partition", "issue", "source_rows", "dest_rows",
                 "source_files", "dest_files", "source_bytes", "dest_bytes"]
    summary_df = pd.DataFrame(_rows).reindex(columns=summary_cols)
    partitions_df = pd.DataFrame(_diffs).reindex(columns=diff_cols)
    for _df, _cols in ((summary_df, summary_cols[4:13]), (partitions_df, diff_cols[4:])):
        for _col in _cols:
            _df[_col] = _df[_col].astype("Int64")

    print()
    print(summary_df["status"].value_counts().to_string())
    display(summary_df)

---
## Step 3: Write the report

Writes `summary.csv`, `partition_diffs.csv` and `report.html` to a new `report_<UTC timestamp>/` folder.

In [ ]:
STATUS_COLORS = {
    "PASS": "#d4edda", "WARN": "#fff3cd", "FAIL": "#f8d7da", "ERROR": "#f8d7da",
    "MISSING_ON_DEST": "#f8d7da", "EXTRA_ON_DEST": "#e2e3e5", "SKIPPED_VIEW": "#e2e3e5",
}


def _html_table(df):
    shown = df.copy()
    for col in [c for c in shown.columns if c.endswith("_bytes")]:
        shown[col] = shown[col].map(lambda n: "" if pd.isna(n) else mmc.fmt_bytes(int(n)))
    out = shown.to_html(index=False, na_rep="", escape=True, border=0)
    for status, color in STATUS_COLORS.items():
        out = out.replace(f"<td>{status}</td>", f'<td style="background:{color};font-weight:bold">{status}</td>')
    return out


def render_report(summary, diffs, src_snap, dest_snap):
    counts = " &nbsp; ".join(f"<b>{html.escape(k)}</b>: {v}" for k, v in summary["status"].value_counts().items())
    shown_note = ""
    if len(diffs) > REPORT_MAX_PARTITION_ROWS:
        shown_note = f", first {REPORT_MAX_PARTITION_ROWS} shown, see partition_diffs.csv"
    return f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>Migration validation: {html.escape(RUN_LABEL)}</title>
<style>
 body {{ font-family: sans-serif; font-size: 13px; margin: 24px; }}
 table {{ border-collapse: collapse; margin-bottom: 24px; }}
 th, td {{ border: 1px solid #ccc; padding: 4px 8px; vertical-align: top; }}
 th {{ background: #f2f2f2; }}
</style></head><body>
<h1>Migration validation: {html.escape(src_snap['label'])} &rarr; {html.escape(dest_snap['label'])}</h1>
<p>Run label <b>{html.escape(RUN_LABEL)}</b>.
 Source collected {html.escape(src_snap['collected_at'])}, dest collected {html.escape(dest_snap['collected_at'])}.</p>
<p>{counts}</p>
<h2>Tables</h2>
{_html_table(summary)}
<h2>Partition differences ({len(diffs):,}{shown_note})</h2>
{_html_table(diffs.head(REPORT_MAX_PARTITION_ROWS)) if len(diffs) else "<p>None.</p>"}
</body></html>"""


if summary_df is None:
    print("No comparison to report; see Step 2.")
else:
    report_dir = f"{RUN_DIR}/report_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
    write_text(f"{report_dir}/summary.csv", summary_df.to_csv(index=False))
    write_text(f"{report_dir}/partition_diffs.csv", partitions_df.to_csv(index=False))
    write_text(f"{report_dir}/report.html", render_report(summary_df, partitions_df, src_snapshot, dest_snapshot))
    print(f"Report written to {report_dir}/")

---
## Drill-down: one table

Lists every partition-level difference for one table: rows per partition, file count and bytes
per partition directory, and partition directories that are not registered.

In [ ]:
TABLE_TO_INSPECT = ""  # table name, e.g. "orders"

if TABLE_TO_INSPECT and summary_df is not None:
    _mask = summary_df["table"].str.lower() == TABLE_TO_INSPECT.lower()
    display(summary_df[_mask].T)
    display(partitions_df[partitions_df["table"].str.lower() == TABLE_TO_INSPECT.lower()])